# Module 11 — Multi-Head Attention

Module 10 ran scaled dot-product attention once, over the whole
`d_model`-dimensional space. **Multi-head attention** splits `d_model` into
several smaller subspaces ("heads"), runs the *same* attention mechanism
independently in each, then concatenates the results back together. Each
head gets to specialize — one might learn to attend mostly to the
immediately preceding token, another to something further back — and
because each head works in a smaller subspace, the total compute stays
about the same as one big head.

## 1. Reusing Module 10's attention function (copied in, not re-explained)

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def scaled_dot_product_attention(Q, K, V, causal=True):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if causal:
        seq_len_q, seq_len_k = scores.shape[-2], scores.shape[-1]
        mask = torch.triu(torch.ones(seq_len_q, seq_len_k), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


torch.manual_seed(42)
seq_len, d_model, num_heads = 6, 8, 2
d_k = d_model // num_heads
x = torch.randn(seq_len, d_model)

Wq = nn.Linear(d_model, d_model, bias=False)
Wk = nn.Linear(d_model, d_model, bias=False)
Wv = nn.Linear(d_model, d_model, bias=False)
Wo = nn.Linear(d_model, d_model, bias=False)

Q, K, V = Wq(x), Wk(x), Wv(x)
print("Q, K, V shape (still full d_model, not yet split into heads):", Q.shape)

## 2. Naive version: a Python loop over heads

Split each of Q/K/V's last dimension into `num_heads` chunks of size
`d_k = d_model / num_heads`, run attention independently per chunk, then
concatenate the outputs back to `d_model`.

In [ ]:
def multi_head_naive(Q, K, V, num_heads):
    d_model = Q.shape[-1]
    d_k = d_model // num_heads
    head_outputs = []
    for h in range(num_heads):
        Qh = Q[:, h * d_k:(h + 1) * d_k]
        Kh = K[:, h * d_k:(h + 1) * d_k]
        Vh = V[:, h * d_k:(h + 1) * d_k]
        out_h, _ = scaled_dot_product_attention(Qh, Kh, Vh, causal=True)
        head_outputs.append(out_h)
    return torch.cat(head_outputs, dim=-1)  # (seq_len, d_model)


concat_naive = multi_head_naive(Q, K, V, num_heads)
output_naive = Wo(concat_naive)
print("naive multi-head output shape:", output_naive.shape)

## 3. Vectorized version: one batched call instead of a loop

Real implementations don't loop in Python — they reshape so the heads
become a batch dimension, and run one batched matrix multiply that computes
every head simultaneously. `scaled_dot_product_attention` from Module 10
already works unmodified here: `@` and `softmax(dim=-1)` operate on the
last two dimensions no matter how many batch dimensions come before them.

In [ ]:
def split_heads(t, num_heads):
    seq_len, d_model = t.shape
    d_k = d_model // num_heads
    return t.view(seq_len, num_heads, d_k).transpose(0, 1)  # (num_heads, seq_len, d_k)


def merge_heads(t):
    num_heads, seq_len, d_k = t.shape
    return t.transpose(0, 1).contiguous().view(seq_len, num_heads * d_k)


Qh, Kh, Vh = split_heads(Q, num_heads), split_heads(K, num_heads), split_heads(V, num_heads)
print("split shape (num_heads, seq_len, d_k):", Qh.shape)

out_vectorized, weights_vectorized = scaled_dot_product_attention(Qh, Kh, Vh, causal=True)
concat_vectorized = merge_heads(out_vectorized)
output_vectorized = Wo(concat_vectorized)

assert torch.allclose(concat_vectorized, concat_naive, atol=1e-6)
assert torch.allclose(output_vectorized, output_naive, atol=1e-6)
print("Vectorized (batched) result matches the naive per-head loop exactly.")

## 4. Confirming each head is still causally masked independently

In [ ]:
for h in range(num_heads):
    future_weight = weights_vectorized[h, 0, 1:].sum().item()
    assert future_weight == 0.0
print(f"All {num_heads} heads correctly assign 0 attention weight to future positions at query position 0.")

## 5. Wrapping it into a reusable `MultiHeadAttention` module

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, causal=True):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        Qh, Kh, Vh = split_heads(Q, self.num_heads), split_heads(K, self.num_heads), split_heads(V, self.num_heads)
        out, _ = scaled_dot_product_attention(Qh, Kh, Vh, causal=causal)
        return self.Wo(merge_heads(out))


mha = MultiHeadAttention(d_model, num_heads)
mha.Wq.load_state_dict(Wq.state_dict())
mha.Wk.load_state_dict(Wk.state_dict())
mha.Wv.load_state_dict(Wv.state_dict())
mha.Wo.load_state_dict(Wo.state_dict())

module_output = mha(x)
assert torch.allclose(module_output, output_vectorized, atol=1e-6)
print("MultiHeadAttention module matches the manual vectorized computation.")

## Recap

- Multi-head attention = split `d_model` into `num_heads` subspaces, run
  the *same* scaled dot-product attention from Module 10 in each, then
  concatenate and project back with `Wo`.
- The naive per-head Python loop and the vectorized batched version compute
  identical results — the vectorized version is just how it's actually
  implemented for speed.
- Causal masking still applies independently within every head.

This `MultiHeadAttention` module is one of the two core pieces of a
transformer block. Modules 12-15 build the other supporting pieces
(positional encoding, layer norm, residual connections, feed-forward), and
Module 16 assembles them all together.